Conectar con google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
INPUT_CSV = '/content/drive/MyDrive/full_dataset.csv'

# Leemos solo la cabecera (nrows=0)
headers = pd.read_csv(INPUT_CSV, nrows=0).columns.tolist()
print("Las columnas reales de tu archivo son:")
print(headers)

Las columnas reales de tu archivo son:
['Unnamed: 0', 'title', 'ingredients', 'directions', 'link', 'source', 'NER']


Limpieza y conversión a parquet

In [ ]:
# ==========================================
# PASO: LIMPIEZA Y CONVERSIÓN A PARQUET
# ==========================================
import pandas as pd
import ast
import time

# RUTAS EXACTAS
INPUT_CSV = '/content/drive/MyDrive/full_dataset.csv'
OUTPUT_PARQUET = '/content/drive/MyDrive/recipes_full_cleaned.parquet'

print(f"🚀 Iniciando procesamiento de {INPUT_CSV}...")
t_start = time.time()

# 1. CARGA OPTIMIZADA (Solo columnas necesarias para no saturar la RAM)
cols = ['title', 'ingredients', 'directions', 'NER']
try:
    # Cargamos el CSV. Nota: Si Colab se queda sin RAM, avisame para usar chunks.
    df = pd.read_csv(INPUT_CSV, usecols=cols)
    print(f"✅ Dataset cargado: {len(df):,} recetas.")
except Exception as e:
    print(f"❌ Error al leer el archivo: {e}")
    raise

# 2. LIMPIEZA DE DATOS (Data Cleaning)
print("🧹 Limpiando el dataset...")

# Eliminar filas con cualquier campo vacío
df.dropna(inplace=True)

# Eliminar recetas con el mismo título (RecipeNLG tiene muchos duplicados)
df.drop_duplicates(subset=['title'], keep='first', inplace=True)

# Filtrar recetas "basura" (ej. las que tienen menos de 2 ingredientes o pasos)
# Usamos una técnica rápida: contar comas antes de convertir a lista
df = df[df['NER'].str.count(',') >= 1]
df = df[df['directions'].str.count(',') >= 1]

print(f"   -> Recetas de alta calidad restantes: {len(df):,}")

# 3. NORMALIZACIÓN DE FORMATOS
print("⚙️ Convirtiendo textos de ingredientes en listas de Python...")
def safe_eval(val):
    try:
        return ast.literal_eval(val)
    except:
        return []

# Esto es lo que permite que tu Retrieval funcione rápido después
df['NER'] = df['NER'].apply(safe_eval)

# 4. GUARDADO EN PARQUET
print(f"💾 Guardando resultado en {OUTPUT_PARQUET}...")
df.to_parquet(OUTPUT_PARQUET, index=False, engine='pyarrow')

t_end = time.time()
print(f"\n🎉 ¡TERMINADO en {t_end - t_start:.2f} segundos!")
print(f"Ahora tienes {len(df):,} recetas listas para usar en tu motor central.")

🚀 Iniciando procesamiento de /content/drive/MyDrive/full_dataset.csv...
✅ Dataset cargado: 2,231,142 recetas.
🧹 Limpiando el dataset...
   -> Recetas de alta calidad restantes: 1,275,861
⚙️ Convirtiendo textos de ingredientes en listas de Python...
💾 Guardando resultado en /content/drive/MyDrive/recipes_full_cleaned.parquet...

🎉 ¡TERMINADO en 107.21 segundos!
Ahora tienes 1,275,861 recetas listas para usar en tu motor central.
